In [3]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [4]:
df_ratings = pd.read_csv("ratings.csv")
df_movies = pd.read_csv("movies.csv")

df = df_ratings.merge(df_movies[['movieId', 'title']], how='left', on='movieId')
# User-Item матрица
df_user_item = df.pivot_table(index='userId', columns='title', values='rating')

print("Размер матрицы user-item:", df_user_item.shape)

corr_matrix = df_user_item.corr(method='pearson', min_periods=100)

user_id = 7
user_ratings = df_user_item.loc[user_id].dropna()

print(f"\nПользователь {user_id} оценил {len(user_ratings)} фильмов.")

sim_candidates = pd.Series(dtype='float64')

for movie in user_ratings.index:
    sims = corr_matrix[movie].dropna()
    sims = sims.map(lambda x: x * user_ratings[movie])
    sim_candidates = pd.concat([sim_candidates, sims])

sim_candidates = sim_candidates.groupby(sim_candidates.index).sum()

not_watched = list(set(sim_candidates.index) - set(user_ratings.index))
filtered_sims = sim_candidates[not_watched].sort_values(ascending=False)


top3 = filtered_sims.head(3)
print("\nТоп-3 рекомендаций для пользователя 7:")
print(top3)

Размер матрицы user-item: (610, 9719)

Пользователь 7 оценил 152 фильмов.

Топ-3 рекомендаций для пользователя 7:
Matrix, The (1999)                                                                30.707095
Raiders of the Lost Ark (Indiana Jones and the Raiders of the Lost Ark) (1981)    22.002139
Shawshank Redemption, The (1994)                                                  18.925514
dtype: float64
